<a href="https://colab.research.google.com/github/Briyad37/Convolutional-Neural-Networks-TomatoDoc-Models/blob/main/perceptual_hash_duplicate_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duplicate Detection and Clean Dataset Creation using Perceptual Hashing

This notebook checks for duplicate/similar images between the training and testing folders using perceptual hashing.

It does **not** modify the original dataset. Instead, it creates a new cleaned dataset folder and saves a duplicate report CSV.

## Goal

Original dataset:

```text
image_dataset/
    train/
    test/
```

Output:

```text
image_dataset/
    clean_dataset_no_duplicates_hashing/
        train/
        test/
    duplicate_report_hashing.csv
```

The training folder is copied fully. The testing folder is copied while skipping test images that duplicate training images.

In [1]:
# 1. Install required package
!pip install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 5.9 MB/s eta 0:00:00


In [2]:
# 2. Imports

import os
import shutil
import pandas as pd
from PIL import Image
import imagehash

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 3. Dataset paths

dataset_directory_location = "/content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset"

training_part = os.path.join(dataset_directory_location, "train")
testing_part = os.path.join(dataset_directory_location, "test")

CLEAN_DATASET_ROOT = os.path.join(
    dataset_directory_location,
    "clean_dataset_no_duplicates_hashing"
)

REPORT_PATH = os.path.join(
    dataset_directory_location,
    "duplicate_report_hashing.csv"
)

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Smaller number = stricter duplicate detection
# 0 means exactly same hash
# 5 means very visually similar
HASH_DISTANCE_THRESHOLD = 5

print("Training folder:", training_part)
print("Testing folder:", testing_part)
print("Clean dataset will be created at:", CLEAN_DATASET_ROOT)

Training folder: /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/train
Testing folder: /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/test
Clean dataset will be created at: /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/clean_dataset_no_duplicates_hashing


In [4]:
# 4. Check if a file is an image

def is_image_file(filename):
    return filename.lower().endswith(IMAGE_EXTENSIONS)

In [5]:
# 5. Collect image paths recursively

def collect_image_paths(folder):
    image_paths = []

    for root, _, files in os.walk(folder):
        for file in files:
            if is_image_file(file):
                full_path = os.path.join(root, file)
                image_paths.append(full_path)

    return image_paths

In [6]:
# 6. Calculate perceptual hash for one image

def calculate_image_hash(image_path):
    try:
        image = Image.open(image_path)
        image_hash = imagehash.phash(image)
        return image_hash

    except Exception as e:
        print(f"Could not hash image: {image_path}")
        print(f"Error: {e}")
        return None

In [7]:
# 7. Build a hash database for many images

def build_hash_database(image_paths):
    hash_database = {}

    for path in image_paths:
        img_hash = calculate_image_hash(path)

        if img_hash is not None:
            hash_database[path] = img_hash

    return hash_database

In [8]:
# 8. Find duplicate images between train and test

def find_cross_folder_duplicates():
    duplicates_found = []

    print("Collecting training images...")
    train_images = collect_image_paths(training_part)

    print("Collecting testing images...")
    test_images = collect_image_paths(testing_part)

    print(f"Training images found: {len(train_images)}")
    print(f"Testing images found: {len(test_images)}")

    print("Calculating hashes for training images...")
    train_hashes = build_hash_database(train_images)

    print("Calculating hashes for testing images...")
    test_hashes = build_hash_database(test_images)

    print("Comparing train and test hashes...")

    for train_path, train_hash in train_hashes.items():
        for test_path, test_hash in test_hashes.items():

            hash_distance = train_hash - test_hash

            if hash_distance <= HASH_DISTANCE_THRESHOLD:
                duplicates_found.append({
                    "train_image": train_path,
                    "test_image": test_path,
                    "hash_distance": hash_distance,
                    "train_filename": os.path.basename(train_path),
                    "test_filename": os.path.basename(test_path)
                })

                print("Duplicate found:")
                print(f"Train: {os.path.basename(train_path)}")
                print(f"Test : {os.path.basename(test_path)}")
                print(f"Hash distance: {hash_distance}")
                print("-" * 40)

    return duplicates_found

In [9]:
# 9. Copy images while preserving folder/class structure

def copy_images_with_structure(source_folder, target_folder, paths_to_skip=None):
    if paths_to_skip is None:
        paths_to_skip = set()

    print(f"Copying from {source_folder}")
    print(f"To: {target_folder}")

    for root, _, files in os.walk(source_folder):
        relative_path = os.path.relpath(root, source_folder)
        target_dir = os.path.join(target_folder, relative_path)

        os.makedirs(target_dir, exist_ok=True)

        for file in files:
            source_file_path = os.path.join(root, file)

            if not is_image_file(file):
                continue

            if source_file_path in paths_to_skip:
                continue

            destination_file_path = os.path.join(target_dir, file)
            shutil.copy2(source_file_path, destination_file_path)

    print("Copying finished.")

In [10]:
# 10. Main cleaning function

def clean_dataset_using_hashing():
    print("Starting duplicate detection using perceptual hashing...")

    duplicates_found = find_cross_folder_duplicates()

    duplicate_report = pd.DataFrame(duplicates_found)
    duplicate_report.to_csv(REPORT_PATH, index=False)

    print(f"Duplicate report saved at: {REPORT_PATH}")

    test_paths_to_skip = set()

    for duplicate in duplicates_found:
        test_paths_to_skip.add(duplicate["test_image"])

    clean_train_path = os.path.join(CLEAN_DATASET_ROOT, "train")
    clean_test_path = os.path.join(CLEAN_DATASET_ROOT, "test")

    os.makedirs(CLEAN_DATASET_ROOT, exist_ok=True)

    print("Creating clean training folder...")
    copy_images_with_structure(training_part, clean_train_path)

    print("Creating clean testing folder...")
    copy_images_with_structure(testing_part, clean_test_path, test_paths_to_skip)

    print("Cleaning completed.")
    print(f"Clean dataset created at: {CLEAN_DATASET_ROOT}")
    print(f"Number of duplicate test images skipped: {len(test_paths_to_skip)}")

In [11]:
# 11. Run the cleaning process

clean_dataset_using_hashing()

Starting duplicate detection using perceptual hashing...
Training images found: 1620
Testing images found: 1014
Calculating hashes for training images...
Calculating hashes for testing images...
Comparing train and test hashes...
Duplicate found:
Train: 1af0bfe1-4bcf-4b8b-be66-5d0953eb647e___GH_HL Leaf 482.2.JPG
Test : cfd491d6-4af5-4728-8f0e-0d330a07174a___GH_HL Leaf 482.2.JPG
Hash distance: 0
----------------------------------------
Duplicate found:
Train: 37aad83b-7ff8-4b35-b3ed-fb8e0f54910b___GH_HL Leaf 342.1.JPG
Test : e786ac89-29fe-47e3-b49e-b9a9ee7edd9d___GH_HL Leaf 342.1.JPG
Hash distance: 0
----------------------------------------
Duplicate report saved at: /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/duplicate_report_hashing.csv
Creating clean training folder...
Copying from /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/train
To: /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/clean_da